# OOF probabilities — Tầng 1A cho stacking

**Mục đích:** sinh **xác suất out-of-fold (OOF)** trên `train` cho 5 base model, để train meta-learner ở bước stacking mà không rò rỉ.

**Vì sao cần OOF?**
- Model fit trên toàn train rồi dự đoán chính train sẽ cho xác suất "quá đẹp" (đã thấy dữ liệu).
- Meta-learner train trên xác suất đó sẽ học sai.
- OOF: mỗi dòng train được dự đoán bởi model **không thấy dòng đó** khi fit.

**Cấu hình:**
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- Mỗi fold: fit scaler + model trên 4 fold, dự đoán fold còn lại.
- Xác suất test = trung bình của 5 model fold.

**Đầu ra** (`artifacts/probs/oof/`):
- `{model}_oof.npy` — (14,306 × 4)
- `{model}_test.npy` — (3,974 × 4)
- `folds.npy` — (14,306,) chỉ số fold của từng dòng

**Thời gian ước tính:** ~75 phút (LightGBM ~42 phút, LinearSVC calibrated ~24 phút, LogReg ~8 phút, SGD/NB nhanh).

In [10]:
# ============================================================
# 1. IMPORT & CẤU HÌNH
# ============================================================
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
SEED = 42
N_SPLITS = 5


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
OOF_DIR = ARTIFACT_DIR / "probs" / "oof"
OOF_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OOF_DIR     :", OOF_DIR)

PROJECT_ROOT: D:\DS102_NguyenKhacHuy_24520679\Final project
OOF_DIR     : D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\probs\oof


In [11]:
# ============================================================
# 2. NẠP DỮ LIỆU
# ============================================================
X_train = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_train.npz").tocsr()
X_test = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_test.npz").tocsr()
y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
pre = joblib.load(ARTIFACT_DIR / "models" / "preprocessors.joblib")

N_DENSE = len(pre["dense_feature_names"])
N_TRAIN = X_train.shape[0]
N_TEST = X_test.shape[0]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
SPLITS = list(skf.split(X_train, y_train))

fold_of_row = np.zeros(N_TRAIN, dtype=np.int8)
for k, (_, va_idx) in enumerate(SPLITS):
    fold_of_row[va_idx] = k
np.save(OOF_DIR / "folds.npy", fold_of_row)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("dense:", N_DENSE, "| n_splits:", N_SPLITS)
for k, (tr, va) in enumerate(SPLITS):
    print(f"  fold {k}: train={len(tr):5d}  val={len(va):5d}  "
          f"val class dist={np.bincount(y_train[va], minlength=4).tolist()}")

X_train: (14306, 98713) | X_test: (3974, 98713)
dense: 52 | n_splits: 5
  fold 0: train=11444  val= 2862  val class dist=[2103, 43, 220, 496]
  fold 1: train=11445  val= 2861  val class dist=[2103, 42, 220, 496]
  fold 2: train=11445  val= 2861  val class dist=[2102, 42, 221, 496]
  fold 3: train=11445  val= 2861  val class dist=[2102, 42, 221, 496]
  fold 4: train=11445  val= 2861  val class dist=[2102, 42, 221, 496]


## 3. Hàm chạy OOF

Mỗi lần gọi tạo **model mới** (factory), fit scaler riêng trên fold train, và:
- ghi xác suất vào đúng vị trí `oof[val_idx]`
- cộng dồn xác suất test, cuối cùng chia cho số fold
- in accuracy từng fold + accuracy OOF tổng

In [12]:
# ============================================================
# 3. HÀM CHẠY OOF
# ============================================================
OOF_RESULTS = {}


def assemble(X_tr, X_va, X_te, tr_idx):
    """Scale dense bằng scaler fit trên fold train, ghép với TF-IDF."""
    scaler = StandardScaler(with_mean=False).fit(X_tr[tr_idx][:, :N_DENSE])
    A_tr = sparse.hstack(
        [scaler.transform(X_tr[tr_idx][:, :N_DENSE]), X_tr[tr_idx][:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_va = sparse.hstack(
        [scaler.transform(X_va[:, :N_DENSE]), X_va[:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_te = sparse.hstack(
        [scaler.transform(X_te[:, :N_DENSE]), X_te[:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    return A_tr, A_va, A_te


def run_oof(name, factory, tfidf_only=False):
    oof = np.zeros((N_TRAIN, 4), dtype=np.float32)
    test_sum = np.zeros((N_TEST, 4), dtype=np.float32)
    fold_acc = []
    t_start = time.time()

    for k, (tr_idx, va_idx) in enumerate(SPLITS):
        t0 = time.time()
        if tfidf_only:
            A_tr = X_train[tr_idx][:, N_DENSE:]
            A_va = X_train[va_idx][:, N_DENSE:]
            A_te = X_test[:, N_DENSE:]
        else:
            A_tr, A_va, A_te = assemble(X_train, X_train[va_idx], X_test, tr_idx)

        model = factory()
        model.fit(A_tr, y_train[tr_idx])
        p_va = model.predict_proba(A_va)
        oof[va_idx] = p_va
        test_sum += model.predict_proba(A_te) / N_SPLITS

        acc = accuracy_score(y_train[va_idx], p_va.argmax(axis=1))
        fold_acc.append(acc)
        print(f"  {name:14s} fold {k}: acc={acc:.4f}  ({time.time() - t0:.0f}s)")

    oof_acc = accuracy_score(y_train, oof.argmax(axis=1))
    print(f"{name:14s} OOF accuracy = {oof_acc:.4f}  | "
          f"trung bình fold = {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}  | "
          f"tổng {time.time() - t_start:.0f}s")

    np.save(OOF_DIR / f"{name}_oof.npy", oof)
    np.save(OOF_DIR / f"{name}_test.npy", test_sum)
    OOF_RESULTS[name] = {
        "oof_accuracy": round(float(oof_acc), 4),
        "fold_mean": round(float(np.mean(fold_acc)), 4),
        "fold_std": round(float(np.std(fold_acc)), 4),
        "seconds": round(float(time.time() - t_start), 1),
    }
    return oof, test_sum

## 4. Logistic Regression

~8 phút cho 5 fold.

In [4]:
oof_logreg, test_logreg = run_oof(
    "logreg",
    lambda: LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=SEED),
)

  logreg         fold 0: acc=0.9015  (97s)
  logreg         fold 1: acc=0.8899  (95s)
  logreg         fold 2: acc=0.8861  (95s)
  logreg         fold 3: acc=0.8868  (91s)
  logreg         fold 4: acc=0.8962  (95s)
logreg         OOF accuracy = 0.8921  | trung bình fold = 0.8921 ± 0.0059  | tổng 474s


## 5. LinearSVC + calibration

~24 phút vì mỗi fold phải calibration 3-fold bên trong.

In [5]:
oof_linsvc, test_linsvc = run_oof(
    "linsvc",
    lambda: CalibratedClassifierCV(
        estimator=LinearSVC(C=0.5, max_iter=5000, random_state=SEED),
        cv=3, method="sigmoid",
    ),
)

  linsvc         fold 0: acc=0.8997  (245s)
  linsvc         fold 1: acc=0.8875  (236s)
  linsvc         fold 2: acc=0.8854  (240s)
  linsvc         fold 3: acc=0.8882  (244s)
  linsvc         fold 4: acc=0.8979  (298s)
linsvc         OOF accuracy = 0.8917  | trung bình fold = 0.8917 ± 0.0059  | tổng 1263s


## 6. SGDClassifier

~1 phút.

In [6]:
oof_sgd, test_sgd = run_oof(
    "sgd",
    lambda: SGDClassifier(
        loss="modified_huber", alpha=1e-5, max_iter=3000, tol=1e-4,
        random_state=SEED,
    ),
)

  sgd            fold 0: acc=0.8697  (6s)
  sgd            fold 1: acc=0.8375  (6s)
  sgd            fold 2: acc=0.8570  (5s)
  sgd            fold 3: acc=0.8591  (6s)
  sgd            fold 4: acc=0.8798  (5s)
sgd            OOF accuracy = 0.8606  | trung bình fold = 0.8606 ± 0.0141  | tổng 29s


## 7. Complement Naive Bayes

Chạy trên TF-IDF thuần (không âm). Gần như tức thời.

In [7]:
oof_cnb, test_cnb = run_oof(
    "complement_nb",
    lambda: ComplementNB(alpha=1.0),
    tfidf_only=True,
)

  complement_nb  fold 0: acc=0.8295  (0s)
  complement_nb  fold 1: acc=0.8308  (0s)
  complement_nb  fold 2: acc=0.8242  (0s)
  complement_nb  fold 3: acc=0.8343  (0s)
  complement_nb  fold 4: acc=0.8270  (0s)
complement_nb  OOF accuracy = 0.8292  | trung bình fold = 0.8292 ± 0.0034  | tổng 1s


## 8. LightGBM

~42 phút cho 5 fold — cell lâu nhất.

In [ ]:
from lightgbm import LGBMClassifier

oof_lightgbm, test_lightgbm = run_oof(
    "lightgbm",
    lambda: LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED,
        n_jobs=-1, verbose=-1
    ),
)

  lightgbm       fold 0: acc=0.9018  (529s)
  lightgbm       fold 1: acc=0.8955  (478s)
  lightgbm       fold 2: acc=0.8937  (532s)
  lightgbm       fold 3: acc=0.8948  (528s)
  lightgbm       fold 4: acc=0.8965  (532s)
lightgbm       OOF accuracy = 0.8965  | trung bình fold = 0.8965 ± 0.0028  | tổng 2599s


## 9. Tổng hợp OOF

Bảng dưới đây là ước lượng công bằng cho từng base model (khác với accuracy trên dev vì đây là trung bình qua 5 fold).

In [9]:
summary = pd.DataFrame(OOF_RESULTS).T.sort_values("oof_accuracy", ascending=False)
print("OOF accuracy theo model:")
print(summary.to_string())
print("\nĐã lưu trong:", OOF_DIR)
for p in sorted(OOF_DIR.glob("*.npy")):
    print(f"  {p.name:28s} {p.stat().st_size / 1e6:.2f} MB")

OOF accuracy theo model:
               oof_accuracy  fold_mean  fold_std  seconds
lightgbm             0.8965     0.8965    0.0028   2599.0
logreg               0.8921     0.8921    0.0059    474.1
linsvc               0.8917     0.8917    0.0059   1262.6
sgd                  0.8606     0.8606    0.0141     29.2
complement_nb        0.8292     0.8292    0.0034      0.9

Đã lưu trong: D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\probs\oof
  complement_nb_oof.npy        0.23 MB
  complement_nb_test.npy       0.06 MB
  folds.npy                    0.01 MB
  lightgbm_oof.npy             0.23 MB
  lightgbm_test.npy            0.06 MB
  linsvc_oof.npy               0.23 MB
  linsvc_test.npy              0.06 MB
  logreg_oof.npy               0.23 MB
  logreg_test.npy              0.06 MB
  sgd_oof.npy                  0.23 MB
  sgd_test.npy                 0.06 MB


## 10. Ghi chú

- Các file `{model}_oof.npy` sẽ là đầu vào của meta-learner; **không** dùng `probs/{model}_dev.npy` để train meta.
- `{model}_test.npy` là trung bình xác suất test qua 5 fold — ổn định hơn dự đoán từ một model đơn.
- `folds.npy` ghi fold của từng dòng để tái lập / kiểm tra.
- Nếu một cell bị gián đoạn, chạy lại riêng cell đó; các model khác không bị ảnh hưởng.